# 15. LCA ライフサイクルアセスメント（Life Cycle Assessment） — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

ライフサイクルアセスメント（LCA）は、技術のライフサイクル全段階（製造・運用・廃棄）の資源消費と環境排出を積み上げ、特性化係数で地球温暖化係数（GWP, CO2当量）に換算して環境インパクトを定量評価する手法である。このノートブックでは、大規模言語モデル（LLM）でタスクを自動化する場合と、従来手法（人手・従来ソフトウェア）で同じタスクを行う場合を、共通の機能単位の下で比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## インベントリと特性化係数の定義

LLMによる自動化と従来手法の段階別インベントリ（機能単位＝対象タスクを1回完了する、kWh換算）と、電力1kWhあたりのCO2排出量（特性化係数）を定義する。LLM側は「製造（データセンター・GPU等）」「訓練（基盤モデルの訓練を配分）」「推論運用」「冷却」「廃棄」の段階を持つ。

In [ ]:
STAGES = ["製造", "訓練", "推論運用", "冷却", "廃棄"]

# LLM自動化のインベントリ(基準ケース)。単位は kWh / タスク。
# 「製造」「訓練」「廃棄」は1タスクあたりに配分した固定費的な値。
# 「推論運用」「冷却」は1タスクの実行で増える変動費的な値。
LLM_INVENTORY_BASE = {
    "製造": 8.0,      # データセンター・GPU/アクセラレータ製造の配分
    "訓練": 30.0,     # 基盤モデル1回の訓練の電力消費を配分
    "推論運用": 4.0,  # 推論1タスクあたりの電力消費(基準値)
    "冷却": 2.0,      # データセンター冷却(推論に付随)
    "廃棄": 1.0,      # ハードウェア廃棄の配分
}
# 製造・訓練・廃棄の合計 = 利用回数で割られる固定費。
LLM_FIXED = (LLM_INVENTORY_BASE['製造']
             + LLM_INVENTORY_BASE['訓練']
             + LLM_INVENTORY_BASE['廃棄'])
# 推論運用・冷却の合計 = 1タスクごとに増える変動費。
LLM_VARIABLE = LLM_INVENTORY_BASE['推論運用'] + LLM_INVENTORY_BASE['冷却']

# 従来手法(人手・従来ソフトウェア)のインベントリ。1タスクあたり kWh。
# 人手作業に伴うPC・オフィス電力等を1タスクに配分した想定値。
CONVENTIONAL_PER_TASK = 18.0

# 特性化係数: 電力1kWhあたりのCO2排出量(電源構成依存の想定値)。
GWP_FACTOR = 0.45    # kg-CO2e / kWh

print("機能単位: 「対象タスク(例: 文書作成・分類)を1回完了する」")
print(f"特性化係数: {GWP_FACTOR} kg-CO2e/kWh (電源構成依存の想定値)")
print(f"LLM 固定費(製造+訓練+廃棄) = {LLM_FIXED:.1f} kWh")
print(f"LLM 変動費(推論運用+冷却)  = {LLM_VARIABLE:.1f} kWh / タスク")

## 段階別 GWP の算出

LLM自動化の総排出は、固定費（製造・訓練・廃棄）を利用回数 N で割った1タスクあたりの値に、変動費（推論運用・冷却）を加えたものである。利用回数 N が大きいほど固定費の1タスクあたり負担が薄まる。各段階のエネルギーに特性化係数を掛けて GWP を算出する関数を定義する。

In [ ]:
def llm_stage_gwp(n_uses):
    """LLM自動化の段階別 GWP(kg-CO2e / タスク)を利用回数 N で算出する。
    固定費段階(製造・訓練・廃棄)は N で割って1タスクに配分する。
    """
    manufacture = LLM_INVENTORY_BASE['製造'] / n_uses * GWP_FACTOR
    training = LLM_INVENTORY_BASE['訓練'] / n_uses * GWP_FACTOR
    inference = LLM_INVENTORY_BASE['推論運用'] * GWP_FACTOR
    cooling = LLM_INVENTORY_BASE['冷却'] * GWP_FACTOR
    disposal = LLM_INVENTORY_BASE['廃棄'] / n_uses * GWP_FACTOR
    return manufacture, training, inference, cooling, disposal


def total_gwp_llm(n_uses):
    """LLM自動化の総GWP(kg-CO2e / タスク)を利用回数 N で算出する。"""
    return sum(llm_stage_gwp(n_uses))


def total_gwp_conventional():
    """従来手法の総GWP(kg-CO2e / タスク)を算出する。"""
    return CONVENTIONAL_PER_TASK * GWP_FACTOR


N_REF = 1000  # 基準ケースの利用回数
llm_m, llm_t, llm_i, llm_c, llm_d = llm_stage_gwp(N_REF)
llm_total = total_gwp_llm(N_REF)
conv_total = total_gwp_conventional()

print(f"[段階別 GWP] (kg-CO2e / タスク, LLMは利用回数 N={N_REF} の基準ケース)")
print("-" * 70)
print(f"  {'段階':<10}{'LLM自動化':>16}")
print(f"  {'製造':<10}{llm_m:>16.4f}")
print(f"  {'訓練':<10}{llm_t:>16.4f}")
print(f"  {'推論運用':<10}{llm_i:>16.4f}")
print(f"  {'冷却':<10}{llm_c:>16.4f}")
print(f"  {'廃棄':<10}{llm_d:>16.4f}")
print("  " + "-" * 30)
print(f"  {'総計':<10}{llm_total:>16.4f}")
print(f"\n  従来手法(人手・従来SW)の総GWP = {conv_total:.4f} kg-CO2e / タスク")

llm_stages = {"製造": llm_m, "訓練": llm_t, "推論運用": llm_i,
              "冷却": llm_c, "廃棄": llm_d}
hotspot = max(llm_stages, key=llm_stages.get)
print(f"\n[ホットスポット] N={N_REF} における最大負荷段階 = 『{hotspot}』")

## ブレークイーブン分析

LLM自動化の総GWPが従来手法と一致する利用回数 N* を二分探索で求める。LLM の総GWPは N の単調減少関数なので二分探索が使える。

In [ ]:
def find_breakeven(n_min=1.0, n_max=100000.0):
    """総GWPが従来手法と一致する利用回数 N* を二分探索で求める。"""
    conv = total_gwp_conventional()
    lo, hi = n_min, n_max
    if total_gwp_llm(n_min) < conv:
        return None  # N_min でも既にLLMが有利
    if total_gwp_llm(n_max) > conv:
        return None  # N_max でもLLMが不利のまま
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if total_gwp_llm(mid) > conv:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


breakeven = find_breakeven()
print("[ブレークイーブン分析] LLMの利用回数 N を変数に")
print("-" * 70)
for n in [1, 5, 20, 100, 1000]:
    lt = total_gwp_llm(n)
    verdict = "LLMが有利" if lt < conv_total else "LLMが不利"
    print(f"  N={n:>5}: LLM総GWP={lt:>9.4f}  (従来={conv_total:.4f}) -> {verdict}")
print("-" * 70)
if breakeven is None:
    print("  探索範囲内でブレークイーブン点なし(常に一方が有利)。")
else:
    print(f"  ブレークイーブン点 N* = {breakeven:.1f} 回")
    print(f"  -> N > {breakeven:.1f} なら LLM自動化が環境的に有利、")
    print(f"     N < {breakeven:.1f} なら基盤モデル訓練・製造の固定費が薄まらず")
    print("     LLM自動化の総排出はかえって従来手法を上回る。")

## 可視化1: ライフサイクル段階別 GWP の積み上げ棒グラフ

LLM自動化（N=基準ケース）と従来手法の段階別 GWP を、積み上げ棒グラフで並置する。どの段階が支配的か（ホットスポット）が一目で分かる。

In [ ]:
stages_en = ["Manufacture", "Training", "Inference", "Cooling", "Disposal"]
llm_vals = [llm_m, llm_t, llm_i, llm_c, llm_d]
conv_vals = [0, 0, conv_total, 0, 0]  # 従来手法は運用相当に一括計上
colors = ["#7fb8d6", "#3f7ca6", "#1f3f5c", "#5a8fb0", "#9fc5dc"]

fig, ax = plt.subplots(figsize=(6.5, 5))
x = [0, 1]
labels = [f"LLM (N={N_REF})", "Conventional"]
bottom_llm = 0.0
bottom_conv = 0.0
for k in range(5):
    ax.bar(x[0], llm_vals[k], 0.55, bottom=bottom_llm, color=colors[k],
           label=stages_en[k])
    ax.bar(x[1], conv_vals[k], 0.55, bottom=bottom_conv, color=colors[k])
    bottom_llm += llm_vals[k]
    bottom_conv += conv_vals[k]
ax.text(x[0], bottom_llm + 0.1, f"{bottom_llm:.2f}", ha="center", fontsize=9)
ax.text(x[1], bottom_conv + 0.1, f"{bottom_conv:.2f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("GWP per task (kg-CO2e)")
ax.set_title("LCA: life-cycle stage GWP (LLM automation vs Conventional)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 可視化2: 利用回数に対する総排出曲線とブレークイーブン点

利用回数 N に対する LLM自動化の総GWP曲線を描き、従来手法の総GWP（一定）と交わるブレークイーブン点 N* を注記する。横軸は対数スケールで表示する。

In [ ]:
n_values = np.logspace(0, 4, 200)  # 1 〜 10000 回
llm_curve = np.array([total_gwp_llm(n) for n in n_values])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_values, llm_curve, label="LLM automation (total GWP)",
        color="#1f3f5c", linewidth=2)
ax.axhline(conv_total, color="#c0504d", linestyle="--",
           label="Conventional method (total GWP)")
if breakeven is not None:
    ax.axvline(breakeven, color="#7f7f7f", linestyle=":")
    ax.plot([breakeven], [conv_total], "o", color="#c0504d", zorder=5)
    ax.text(breakeven * 1.15, conv_total * 1.15,
            f"break-even N* = {breakeven:.0f}", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Number of automated uses N (log scale)")
ax.set_ylabel("Total GWP per task (kg-CO2e)")
ax.set_title("LCA: total emissions vs number of LLM uses")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文においてライフサイクルアセスメント（LCA）は、ある技術の環境負荷を、原料採取から廃棄に至る全段階にわたって定量的に積み上げ、代替技術と比較するために用いられる。論文はまず機能単位を定め、システム境界を画定し、段階別のインベントリに特性化係数を掛けて環境影響を集計する。そこで生まれる結論は、ライフサイクル境界内の定量比較という明快な型をとる。「この技術の環境負荷は代替技術よりXだけ低い（あるいは高い）」「ある性能優位の倍率を超えると総排出が逆転する」というブレークイーブンの言明が、数値の裏づけとともに提示される。

この手法が結論に持ち込む規定力は、何よりも境界設定にある。機能単位の定義とシステム境界の引き方が、比較の土俵そのものを決め、ひいては結論の符号すら左右する。境界の内側に入れた段階の負荷だけが集計され、境界の外に置かれた影響——とりわけ社会的・経済的な帰結や、雇用や分配への作用——は構造的に不可視となる。LCA は環境という一領域に焦点を絞ることで定量的な厳密さを獲得するが、その厳密さは射程の狭さと引き換えに得られている。

時間観の面では、LCA は将来の運用条件——電力構成の脱炭素化の進み方など——を前提値として固定する。未来を選択や設計の対象としてではなく、係数として与えられたものとして扱うため、結論はその前提の妥当性に強く依存する。価値の所在は、特性化係数とどの環境影響カテゴリを採るかという選択に埋め込まれる。地球温暖化係数だけを見るか、資源枯渇や生態系影響まで含めるかで、同じ技術の評価は変わりうる。したがってこの手法を用いた論文は、数値の客観性という外観を持ちながら、境界・前提・係数の選択に判断が凝縮されており、その透明な開示とパラメータ感度分析を欠けば、結論は限定された土俵での部分最適を全体の優劣であるかのように語ってしまう。

## 発展課題

**課題A**: 基盤モデルの訓練電力 `LLM_INVENTORY_BASE['訓練']` や特性化係数 `GWP_FACTOR`、推論1タスクあたりの電力などのパラメータを変化させてブレークイーブン点 N* がどう動くか感度分析せよ。結論（LLM自動化が有利になる条件）が前提にどれだけ依存するかを論ぜよ。

**課題B**: 機能単位を「効率倍率」で定義し直せ。LLM自動化が従来手法に対して何倍の効率（同じ時間で処理できるタスク数）を持つかを変数 R とし、従来手法側の排出を R 倍に換算したうえで総排出が逆転する条件を求めよ。また訓練の電力を再エネで賄った場合（訓練段階の特性化係数を下げる）の効果も評価せよ。